# Model Independent Review: Chinese Equities

**Main Findings**
1) *Economic Data not lagged properly:*
- trade balance, and economic activit can't simply be adjusted by .shift()
- it's important to get the exact release date (not reference period) to avoid mismat
- PMI data implemented using release date from original excel file

2) *ETF data not suitable for analysis:*
- EWZ price is not total return (e.g., missing dividend payments, ETF borrowing cost, etc...)
- Clear way to demonstrate ability to forecast equity returns is to use liquid IBOV futures (BM&F/B3)

3) ***»»» Critical Look-ahead bias «««*** :*
- Function denoise_data assumes future information is already known by centering moving average
    - When data.rolling(window=window, **center=True**).mean()
    - Missing crutial walk-forward technic to avoid leaking future information into the backtest

4) *Strategy returns are calculated without .shift(1):*
- If model suggests a position of -1, this position should be held for 1-month
- Return of Dec/2024 signal should be calculated with the price of Jan/2025
- Current implementation uses end-of-month information to take a position in the beginning of the month

5) *Hodrick-Prescott (HP) filter:*
- I changed the methodology to make every model OOS, but HP filter suffers from endpoint bias
- It could be useful to explore other models to increase estimators reliability

## Setup

### Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from functools import reduce
from itertools import product
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg

### Constants

In [ ]:
security_id = 'CH1'
date_start = '2010-01-02'
date_end = '2024-12-31'
min_months = 3*12 # MINIMUM DATA TO CALIBRATE ESTIMATORS

# USE FUTURE DATA? SHOULD BE FALSE
use_future_data = False

# TAKE POSITION WITH FUTURE SIGNAL? SHOULD BE FALSE
use_future_signal = False

### Data

#### Price Data

In [ ]:
price_data = pd.read_csv('China Rolled Return.csv').set_index('date')
price_data.index = pd.to_datetime(price_data.index)
price_data.index.name = 'date'
price_data.tail(5)

In [ ]:
# Create dataframe with monthly total [price + dividends - funding costs (except margin call expenses)] returns
match security_id:
    case 'CH1':
        monthly_data = price_data.loc[:,"China"].copy()
        monthly_data = monthly_data.resample('M').last().pct_change(fill_method=None)[date_start:date_end].to_frame('monthly_total_return')

monthly_data.tail(5)

In [ ]:
monthly_data.to_csv('monthly_data.csv')

#### Economic Data

In [ ]:
econ_data = pd.read_csv('china-data.csv')
econ_data.index = econ_data['release_date']
econ_data.index = pd.to_datetime(econ_data.index)
econ_data.index.name = 'date'
econ_data.tail(5)

In [ ]:
econ_data2 = pd.read_csv('data_indicator_review.csv')
econ_data2.index = pd.to_datetime(econ_data2['release_date'])
econ_data2.index.name = 'date'
econ_data2.tail(5)

## Functions

### Data processing

In [ ]:
def adjust_seasonality(data, columns, period=12, model='additive', fill_method='ffill'):
    """
    Removes the seasonal component from specified columns in a time series DataFrame.

    Parameters:
    ----------
    data : pd.DataFrame
        A pandas DataFrame with datetime index and time series columns.
    
    columns : list of str
        List of column names in `data` to apply seasonal adjustment to.
    
    period : int, default=12
        The number of observations per cycle (e.g., 12 for monthly data with yearly seasonality).
    
    model : str, default='additive'
        Type of seasonal decomposition. Must be either 'additive' or 'multiplicative'.
    
    fill_method : str, default='ffill'
        Method used to fill missing values. Options include 'ffill', 'bfill', etc.

    Returns:
    -------
    adjusted_data : pd.DataFrame
        DataFrame of the same shape as the selected columns, with seasonal component removed.
    
    Raises:
    ------
    ValueError:
        If any column in `columns` is not present in the input `data`.

    Notes:
    -----
    - Infinite values are replaced with 0.
    - Missing values are filled using the specified method, and any remaining NaNs are filled with 0.
    - Assumes the input index is datetime-like and regularly spaced.

    Example:
    -------
    >>> adjusted = adjust_seasonality(df, ['sales', 'temperature'], period=12, model='additive')
    """
    adjusted_data = pd.DataFrame(index=data.index)

    for col in columns:
        if col not in data.columns:
            raise ValueError(f"Column '{col}' not found in the input DataFrame.")
        
        # Clean and prepare the series
        series = data[col].copy()
        series.replace([float('inf'), -float('inf')], np.nan, inplace=True)
        series = series.fillna(method=fill_method).dropna()

        # Seasonal decomposition and adjustment
        decomposition = seasonal_decompose(series, model=model, period=period, extrapolate_trend='freq')
        adjusted_data[col] = series - decomposition.seasonal
    
    return adjusted_data

def denoise_data(data, window=3):
    """
    Smooths time series data using a centered moving average to reduce noise.

    Parameters:
    ----------
    data : pd.DataFrame or pd.Series
        Input time series data. Should have a datetime-like index.
    
    window : int, default=3
        Size of the moving average window. Must be an odd integer to ensure centering.
    
    Returns:
    -------
    denoised_data : pd.DataFrame or pd.Series
        Smoothed version of the input data, with NaNs at the edges filled forward and backward.

    Notes:
    -----
    - Uses a centered moving average (`center=True`), which requires sufficient data before and after each point.
    - Edge NaNs from the rolling operation are filled with backward and forward fill.
    - For optimal centering, `window` should be odd; even windows can lead to minor time shifts.

    Example:
    -------
    >>> denoised = denoise_data(df['temperature'], window=5)
    """
    denoised_data = data.rolling(window=window, center=True).mean()
    denoised_data = denoised_data.fillna(method='bfill').fillna(method='ffill')
    
    return denoised_data

def standardize_data(data):
    """
    Standardizes numeric features in the DataFrame using z-score normalization.

    Each column is transformed to have mean = 0 and standard deviation = 1.

    Parameters:
    ----------
    data : pd.DataFrame
        A DataFrame of numeric features to standardize. All columns must be numeric.

    Returns:
    -------
    standardized_data : pd.DataFrame
        A DataFrame with the same shape and index as `data`, but with standardized values.

    Notes:
    -----
    - Non-numeric columns should be removed or converted prior to using this function.
    - Missing values (NaNs) in the input will propagate through; they are not handled internally.
    - Uses `StandardScaler` from `sklearn.preprocessing`.

    Example:
    -------
    >>> standardized = standardize_data(df[['sales', 'temperature']])
    """
    scaler = StandardScaler()
    standardized_data = pd.DataFrame(
        scaler.fit_transform(data),
        index=data.index,
        columns=data.columns
    )
    return standardized_data

### Statistical Methods

In [ ]:
def compute_principal_component(data, n_components=1):
    """
    Computes the top principal components of the input dataset using PCA.

    Parameters:
    ----------
    data : pd.DataFrame
        Input DataFrame containing numeric features. Should be standardized beforehand.
    
    n_components : int, default=1
        The number of principal components to compute.

    Returns:
    -------
    principal_df : pd.DataFrame
        A DataFrame containing the computed principal components, with index reset.
        Columns are named 'Principal_Component_1', 'Principal_Component_2', etc.

    Notes:
    -----
    - Input `data` should be standardized (e.g., using `StandardScaler`) before applying PCA.
    - PCA is sensitive to scaling and assumes no missing values.
    - The returned DataFrame has the index reset to default (0, 1, 2, ...).

    Example:
    -------
    >>> pca_df = compute_principal_component(standardized_data, n_components=2)
    """
    pca = PCA(n_components=n_components)
    principal_components = pca.fit_transform(data)
    
    principal_df = pd.DataFrame(
        principal_components,
        index=data.index,
        columns=[f'Principal_Component_{i+1}' for i in range(n_components)]
    )
    
    principal_df.reset_index(inplace=True)
    return principal_df

def calculate_annualized_return(group):
    """
    Calculates the annualized return from a series of daily returns.

    Parameters:
    ----------
    group : pd.DataFrame
        DataFrame containing a 'daily_return' column (e.g., in decimal form, not percent).

    Returns:
    -------
    annualized_return : float
        The annualized return assuming 252 trading days per year.

    Notes:
    -----
    - Assumes daily compounding.
    - Assumes all rows in `group` are consecutive trading days.
    """
    total_return = (1 + group['daily_return']).prod()
    total_days = len(group)
    annualized_return = total_return ** (252 / total_days) - 1
    return annualized_return

def walk_forward_ar_backtest(returns: pd.Series, n_ar: int = 12, initial_window: int = 36) -> pd.Series:
    """
    Perform a walk-forward backtest using an AutoRegressive (AR) model.

    This function simulates a forecasting strategy where at each step in time,
    a model is trained on all data available up to that point (starting from `initial_window`)
    and then used to generate a one-step-ahead forecast. The process continues forward,
    creating a series of forecasted signals.

    Parameters:
    ----------
    returns : pd.Series
        A time series of returns (or other univariate data), indexed by datetime.
    n_ar : int, optional
        Number of lags to use in the AutoRegressive model (default is 12).
    initial_window : int, optional
        Minimum number of initial observations to begin the backtest (default is 36).

    Returns:
    -------
    pd.Series
        A time series of predicted (forecasted) values from the AR model,
        indexed by the time of the forecast.
    """

    forecast_signals = []

    # Ensure the time series is sorted by datetime
    returns = returns.sort_index()

    # Check that the index is a DatetimeIndex
    if not isinstance(returns.index, pd.DatetimeIndex):
        raise ValueError("Index must be a DatetimeIndex.")

    # Walk-forward backtest: iterate through each time step from initial_window onward
    for t in range(initial_window, len(returns) + 1):
        # Use all data up to time t (exclusive) as training data
        train_series = returns.iloc[:t]

        try:
            # Fit an AutoRegressive model with specified number of lags
            model = AutoReg(train_series, lags=n_ar, old_names=False).fit()

            # Generate a one-step-ahead forecast
            predicted = model.predict(start=len(train_series), end=len(train_series))

            # Extract the forecasted signal value
            #signal = predicted.iloc[0]
            signal = predicted[0]
        except Exception:
            # If the model fails to fit or predict, assign NaN and skip to next step
            signal = np.nan
            continue

        # Append the forecasted signal with the corresponding date
        forecast_signals.append([train_series.index[-1], signal])

    # Create a DataFrame of signals and set the date as the index
    df_signal = pd.DataFrame(forecast_signals, columns=['date', 'signal']).set_index('date')

    # Return the signal column as a Series
    return df_signal['signal']

def ir(excess_returns: pd.DataFrame, fnorm=252, pct=True, precision=6) -> pd.DataFrame:
    """
    Calculate the annualized mean, standard deviation, and information ratio (IR) 
    for a given DataFrame or Series of returns.

    Parameters:
    excess_returns (pd.DataFrame or pd.Series): Input data containing returns (typically daily).
    fnorm (int): Normalization factor to annualize statistics (default is 252 trading days).

    Returns:
    pd.DataFrame: A DataFrame containing the annualized mean, standard deviation, 
                  and information ratio for each column in `x`.
    """
    # Calculate the mean and standard deviation of the input data
    df_tmp = excess_returns.agg(['mean', 'std'])

    # Annualize the mean by multiplying by the number of periods in a year (e.g., 252 for daily)
    df_tmp.loc['mean'] *= fnorm

    # Annualize the standard deviation by multiplying by the square root of the number of periods
    df_tmp.loc['std'] *= (fnorm ** 0.5)
    
    # Calculate the Information Ratio: annualized mean divided by annualized standard deviation
    df_tmp.loc['ir'] = df_tmp.loc['mean'] / df_tmp.loc['std']

    # Transform mean and std to %
    if pct:
        df_tmp = df_tmp.rename({'mean':r'%_mean', 'std':r'%_std'})
        df_tmp.loc[[r'%_mean',r'%_std']] *= 100
    return df_tmp.round(precision)

def classify_regime(row, neutral_range=5):
    if abs(row['level']) < neutral_range:
        return 'neutral'
    elif row['level'] >= neutral_range and row['delta'] >= 0:
        return 'expansion'
    elif row['level'] >= neutral_range and row['delta'] < 0:
        return 'normalization'
    elif row['level'] <= -neutral_range and row['delta'] < 0:
        return 'contraction'
    elif row['level'] <= -neutral_range and row['delta'] >= 0:
        return 'recovery'
    else:
        return 'NA'

### Plot Functions

In [ ]:
def plot_factor(data, column, title, xlabel, ylabel):
    """
    Plots a time series column from a DataFrame against a 'date' column.

    Parameters:
    ----------
    data : pd.DataFrame
        A DataFrame containing a 'date' column and the target `column` to plot.
    
    column : str
        Name of the column to plot on the y-axis.
    
    title : str
        Plot title.
    
    xlabel : str
        Label for the x-axis.
    
    ylabel : str
        Label for the y-axis.

    Returns:
    -------
    None
        Displays the plot using matplotlib.

    Notes:
    -----
    - If `data['date']` is a Period type, it is converted to timestamp.
    - The function modifies the original DataFrame by setting 'date' as the index.
    - The plot uses a fixed blue line, with grid and legend enabled.

    Example:
    -------
    >>> plot_factor(df, column='sales', title='Monthly Sales', xlabel='Date', ylabel='Sales')
    """
    # Convert Period to Timestamp if needed
    if isinstance(data['date'].dtype, pd.PeriodDtype):
        data['date'] = data['date'].dt.to_timestamp()

    # Set index to date (modifies original DataFrame)
    data.set_index('date', inplace=True)

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.plot(data.index, data[column], color='blue', label=column)
    plt.title(title, fontsize=14)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.legend()
    plt.grid(True)
    plt.show()

    # Set variable name (modifies original DataFrame)
    data.rename({'Principal_Component_1':ylabel}, axis=1, inplace=True)
    return

def plot_bar(trend_summary, title, xlabel, ylabel, colors=None, figsize=(10, 6)):
    """
    Plots a simple bar chart from a Series or DataFrame summary.

    Parameters:
    ----------
    trend_summary : pd.Series or pd.DataFrame
        Data with categorical index (e.g., labels) and numeric values.
    
    title : str
        Title of the chart.
    
    xlabel : str
        Label for the x-axis.
    
    ylabel : str
        Label for the y-axis.
    
    colors : list of str, optional
        List of colors for the bars. Defaults to ['blue', 'orange'].
    
    figsize : tuple, default=(10, 6)
        Figure size in inches.

    Returns:
    -------
    None
        Displays a matplotlib bar chart.
    """
    categories = trend_summary.index
    values = trend_summary.values

    if colors is None:
        colors = ['blue', 'orange']

    plt.figure(figsize=figsize)
    plt.bar(categories, values, color=colors[:len(categories)])
    plt.title(title, fontsize=14)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.grid(alpha=0.3)
    plt.show()

def plot_box(data, x_col, y_col, title=None, xlabel=None, ylabel=None, palette='Set2', figsize=(10, 6)):
    """
    Draws a box plot comparing distributions of y_col across categories in x_col.

    Parameters:
    ----------
    data : pd.DataFrame
        The dataset containing the x and y columns.
    
    x_col : str
        The column to be used as the category axis (x-axis).
    
    y_col : str
        The column to be used for the value axis (y-axis).
    
    title : str, optional
        Title of the plot.
    
    xlabel : str, optional
        Custom label for the x-axis.
    
    ylabel : str, optional
        Custom label for the y-axis.
    
    palette : str or list, default='Set2'
        Color palette for the boxplot.
    
    figsize : tuple, default=(10, 6)
        Size of the figure in inches.

    Returns:
    -------
    None
        Displays the Seaborn box plot.
    """
    plt.figure(figsize=figsize)
    sns.boxplot(data=data, x=x_col, y=y_col, palette=palette)

    if title:
        plt.title(title, fontsize=14)
    if xlabel:
        plt.xlabel(xlabel, fontsize=12)
    if ylabel:
        plt.ylabel(ylabel, fontsize=12)

    plt.show()

## Core Model

### Factor Construction

#### Domestic Economy Factor

In [ ]:
# 确保 value 列为数值型
econ_data['value'] = pd.to_numeric(econ_data['value'], errors='coerce')

dom_econ_variables = ['china_non-industrial_PMI', 'china_industrial_PMI', 'china-real_estate-climate']
ylabel = 'Domestic Economic Factor'
nickname = 'dom_econ'

dom_econ = econ_data.query("indicator in @dom_econ_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
# 只保留数值型列，防止聚合时报错
dom_econ = dom_econ.select_dtypes(include=[np.number])
dom_econ = dom_econ.query('date >= @date_start and date <= @date_end').copy()
dom_econ.tail(5)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

if use_future_data:
    # Incorrect implementation leaking future data
    dom_econ_factor = compute_principal_component(
        standardize_data(
            denoise_data(
                adjust_seasonality(dom_econ, dom_econ_variables, period=12), 
                window=3
            )
        ), 
        n_components=1
    )
else:
    # Out-of-sample (OOS) walk-forward methodology
    dom_econ_factor = []
    for i in range(min_months, dom_econ.shape[0]+1):
        df_tmp = compute_principal_component(
            standardize_data(
                denoise_data(
                    adjust_seasonality(
                        dom_econ.iloc[:i].fillna(dom_econ.iloc[:i].mean()), 
                        dom_econ_variables, period=12
                    ), 
                    window=3
                )
            ), 
            n_components=1
        )
        dom_econ_factor.append(df_tmp.iloc[[-1]])
    dom_econ_factor = pd.concat(dom_econ_factor, axis=0)

# Plot OOS Factor
plot_factor(dom_econ_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
dom_econ_factor['trend'] = np.where(dom_econ_factor[ylabel].diff()>=0, 'Uptrend', 'Downtrend')
dom_econ_factor[f'{nickname}_signal'] = np.where(dom_econ_factor['trend'] == 'Uptrend', 1, 
                              np.where(dom_econ_factor['trend'] == 'Downtrend', -1, 0))
dom_econ_factor = pd.concat([dom_econ_factor, monthly_data], axis=1).sort_index().dropna()
dom_econ_factor['next_monthly_total_return'] = dom_econ_factor['monthly_total_return'].shift(-1)
dom_econ_factor.tail(5)
dom_econ_factor.to_csv('dom_econ_factor.csv')

In [ ]:
#dom_econ_factor.tail(5)
#dom_econ_factor.to_csv('dom_econ_factor_trend.csv')

In [ ]:
# Select which return to use
if use_future_signal:
    return_label = 'monthly_total_return'
else:
    return_label = 'next_monthly_total_return'

fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=dom_econ_factor, x=return_label, hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title(f'KDE Plot of {return_label} by Trend')
_ = ((dom_econ_factor['dom_econ_signal'] * dom_econ_factor[return_label])
         .shift(1).fillna(0.).add(1).cumprod().plot(title=f'{ylabel}: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Domestic Currency Factor

In [ ]:
dom_curncy_variables = ['china_1Ybond','china_indicator_reserve-ration','china_market-operations','china-3M-shibor','china-1Y-interbank-dposit']
ylabel = 'Domestic Currency Factor'
nickname = 'dom_curncy'

dom_curncy = econ_data.query("indicator in @dom_curncy_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
dom_curncy = dom_curncy.query('date >= @date_start and date <= @date_end').copy()
dom_curncy.tail(5)

In [ ]:
# Out-of-sample (OOS) walk-forward methodology
dom_curncy_factor = []
for i in range(min_months, dom_curncy.shape[0]+1):
    df_tmp = -1 * (
        denoise_data(
            dom_curncy.iloc[:i].fillna(dom_curncy.iloc[:i].mean()), 
            window=3
        ).diff(1).apply(np.sign)
    )
    dom_curncy_factor.append(df_tmp.iloc[[-2]])
dom_curncy_factor = pd.concat(dom_curncy_factor, axis=0)
dom_curncy_factor['dom_curncy_signal'] = dom_curncy_factor[dom_curncy_variables].mean(axis=1).apply(np.sign)

# Plot OOS Factor
plot_factor(dom_curncy_factor.reset_index(), 'dom_curncy_signal', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
dom_curncy_factor = pd.concat([dom_curncy_factor, monthly_data], axis=1).sort_index().dropna()
dom_curncy_factor['trend'] = dom_curncy_factor['dom_curncy_signal'].map({-1: 'Downtrend', 0: 'Neutral', 1:'Uptrend'})
dom_curncy_factor['next_monthly_total_return'] = dom_curncy_factor['monthly_total_return'].shift(-1)
dom_curncy_factor.tail(5)

In [ ]:
#dom_curncy_factor.tail(5)
#dom_curncy_factor.to_csv('dom_curncy_factor_trend.csv')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=dom_curncy_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((dom_curncy_factor['dom_curncy_signal'] * dom_curncy_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Domestic Credit Factor

In [ ]:
dom_credit_variables = ['china_medium_long-term_loans']
ylabel = 'Domestic Credit Factor'
nickname = 'dom_credit'

dom_credit = econ_data.query("indicator in @dom_credit_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
dom_credit = dom_credit.query('date >= @date_start and date <= @date_end').copy()
print(dom_credit.columns)
# Build credit metrics
dom_credit['china_medium_long-term_loans_QoQ'] = -1 * dom_credit['china_medium_long-term_loans'].diff(3)
dom_credit_variables = ['china_medium_long-term_loans_QoQ']
dom_credit = dom_credit[dom_credit_variables].copy()
dom_credit.tail(5)

In [ ]:
# Out-of-sample (OOS) walk-forward methodology
dom_credit_factor = []
for i in range(min_months, dom_credit.shape[0]+1):
    df_tmp = compute_principal_component(
        standardize_data(
            denoise_data(
                adjust_seasonality(
                    dom_credit.iloc[:i].fillna(dom_credit.iloc[:i].mean()), 
                    dom_credit_variables, period=12
                ), 
                window=3
            )
        ), 
        n_components=1
    )
    dom_credit_factor.append(df_tmp.iloc[[-1]])
dom_credit_factor = pd.concat(dom_credit_factor, axis=0)

# Plot OOS Factor
plot_factor(dom_credit_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
dom_credit_factor['trend'] = np.where(dom_credit_factor[ylabel].diff()>=0, 'Uptrend', 'Downtrend')
dom_credit_factor[f'{nickname}_signal'] = np.where(dom_credit_factor['trend'] == 'Uptrend', 1, 
                              np.where(dom_credit_factor['trend'] == 'Downtrend', -1, 0))
dom_credit_factor = pd.concat([dom_credit_factor, monthly_data], axis=1).sort_index().dropna()
dom_credit_factor['next_monthly_total_return'] = dom_econ_factor['monthly_total_return'].shift(-1)
dom_credit_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=dom_credit_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((dom_credit_factor['dom_credit_signal'] * dom_credit_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Domestic Inflation Factor

In [ ]:
dom_inflation_variables = ['china_inflation_PMI']
ylabel = 'Domestic Inflation Factor'
nickname = 'dom_inflation'

dom_inflation = econ_data.query("indicator in @dom_inflation_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
dom_inflation = dom_inflation.query('date >= @date_start and date <= @date_end').copy()

dom_inflation_variables = ['china_inflation_PMI']
dom_inflation = dom_inflation[dom_inflation_variables].copy()
dom_inflation.tail(5)

In [ ]:
# Out-of-sample (OOS) walk-forward methodology
dom_inflation_factor = []
for i in range(min_months, dom_inflation.shape[0]+1):
    df_tmp = compute_principal_component(
        denoise_data(
            adjust_seasonality(
                dom_inflation.iloc[:i].fillna(dom_inflation.iloc[:i].mean()),
                dom_inflation_variables, period=12
                ).rolling(12).sum(),
            window=3
        ),
        n_components=1
    )
    dom_inflation_factor.append(df_tmp.iloc[[-1]])
dom_inflation_factor = pd.concat(dom_inflation_factor, axis=0)

# Plot OOS Factor
plot_factor(dom_inflation_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
# Build ranges
dom_inflation_factor['z'] = ((dom_inflation_factor[ylabel] - dom_inflation_factor[ylabel].ewm(halflife=12,min_periods=12).mean()) /
                                 dom_inflation_factor[ylabel].ewm(halflife=12,min_periods=12).std()
                            ) * (12**0.5)
# Build trend
dom_inflation_factor['trend'] = 'Neutral'
idx = dom_inflation_factor['z'] > 1
dom_inflation_factor.loc[idx, 'trend'] = 'Uptrend'
idx = dom_inflation_factor['z'] < -1
dom_inflation_factor.loc[idx, 'trend'] = 'Downtrend'
# Build signal
dom_inflation_factor['dom_inflation_signal'] = dom_inflation_factor['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
dom_inflation_factor['dom_inflation_signal'] = dom_inflation_factor['dom_inflation_signal'].astype(int)
# Combine with returns
dom_inflation_factor = pd.concat([dom_inflation_factor, monthly_data], axis=1).sort_index().dropna()
dom_inflation_factor['next_monthly_total_return'] = dom_inflation_factor['monthly_total_return'].shift(-1)
dom_inflation_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=dom_inflation_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((dom_inflation_factor['dom_inflation_signal'] * dom_inflation_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

In [ ]:
global_econ_variables = ['usa_pmi','china_pmi_new_export_orders','korea_exports','copper_lme','gold']
ylabel = 'Global Economic Factor'
nickname = 'global_econ'

global_econ = econ_data2.query("indicator in @global_econ_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
global_econ = global_econ.query('date >= @date_start and date <= @date_end').copy()
# Build global_econ metrics
global_econ[['usa_pmi','china_pmi_new_export_orders']] -= 50
global_econ['korea_exports'] = global_econ['korea_exports'].pct_change(12)
global_econ['copper/gold'] = (global_econ['copper_lme'] / global_econ['gold']).apply(np.log)
global_econ_variables = ['usa_pmi','china_pmi_new_export_orders','korea_exports','copper/gold']
global_econ = global_econ[global_econ_variables].copy()
global_econ.tail(5)

In [ ]:
# Out-of-sample (OOS) walk-forward methodology
global_econ_factor = []
for i in range(min_months, global_econ.shape[0]+1):
    df_tmp = compute_principal_component(
        standardize_data(
            denoise_data(
                adjust_seasonality(
                    global_econ.iloc[:i].fillna(global_econ.iloc[:i].mean()),
                    global_econ_variables, period=12
                    ),
                window=3
            ),
        ),
        n_components=1
    )
    global_econ_factor.append(df_tmp.iloc[[-1]])
global_econ_factor = pd.concat(global_econ_factor, axis=0)

# Plot OOS Factor
plot_factor(global_econ_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
# Build trend
global_econ_factor['diff'] = global_econ_factor.diff()
global_econ_factor['trend'] = 'Neutral'
idx = global_econ_factor['diff'] > 0
global_econ_factor.loc[idx, 'trend'] = 'Uptrend'
idx = global_econ_factor['diff'] < 0
global_econ_factor.loc[idx, 'trend'] = 'Downtrend'
# Create signal
global_econ_factor['global_econ_signal'] = global_econ_factor['trend'].map({'Downtrend':-1, 'Neutral':0, 'Uptrend':1})
# Combine with returns
global_econ_factor = pd.concat([global_econ_factor, monthly_data], axis=1).sort_index().dropna()
global_econ_factor['next_monthly_total_return'] = global_econ_factor['monthly_total_return'].shift(-1)
global_econ_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=global_econ_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((global_econ_factor['global_econ_signal'] * global_econ_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Global Currency Factor

In [ ]:
global_curncy_variables = ['treasury_yld_1y', 'fed_securities_bs']
ylabel = 'Global Currency Factor'
nickname = 'global_curncy'
global_curncy = econ_data2.query("indicator in @global_curncy_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
global_curncy = global_curncy.query('date >= @date_start and date <= @date_end').copy()
# Build global_curncy variables
global_curncy['smoothed_yield'] = global_curncy['treasury_yld_1y'].rolling(window=3).mean()
global_curncy['rate_change'] = (global_curncy['smoothed_yield'] - global_curncy['smoothed_yield'].shift(1)).fillna(0.)
global_curncy['yield_signal'] = (global_curncy['rate_change']
                                     .apply(lambda x: np.nan if abs(x) < 0.03 else np.sign(x))
                                     .ffill().fillna(0.).astype(int)
                                )
global_curncy['fed_securities_bs'] /= 1e3 # now in Billions
global_curncy['holdings_signal'] = 2*(global_curncy['fed_securities_bs'].diff()>30)-1
global_curncy_variables = ['smoothed_yield','yield_signal','holdings_signal']
global_curncy = global_curncy[global_curncy_variables].copy()
# Create signal
global_curncy['global_curncy_signal'] = global_curncy.apply(
                                            lambda row: row['holdings_signal'] 
                                            if row['smoothed_yield'] < 0.5
                                            else row['yield_signal'], axis=1
                                        )
global_curncy['trend'] = global_curncy['global_curncy_signal'].map({-1:'Downtrend', 0:'Neutral', 1:'Uptrend'})
# Combine with returns
global_curncy_factor = pd.concat([global_curncy, monthly_data], axis=1).sort_index().dropna()
global_curncy_factor['next_monthly_total_return'] = global_curncy_factor['monthly_total_return'].shift(-1)
global_curncy_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=global_curncy_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0], warn_singular=False)
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((global_curncy_factor['global_curncy_signal'] * global_curncy_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Global Inflation Factor

In [ ]:
global_inflation_variables = ['us_10y_breakeven_inflation']
ylabel = 'Global Inflation Factor'
nickname = 'global_inflation'
global_inflation = econ_data2.query("indicator in @global_inflation_variables").pivot_table('value','date','indicator').sort_index()
global_inflation = global_inflation.query('date >= @date_start and date <= @date_end').sort_index().copy()
# Smooth data with HP filter (OOS)
df_tmp = []
idx = global_inflation.resample('M').last().index
for i, dt in enumerate(idx[12:]):
    trend = (hpfilter(global_inflation[['us_10y_breakeven_inflation']]
                      .query("date <= @dt"), lamb=129_600)[1].iloc[-1]
            )
    df_tmp.append((dt, trend))
global_inflation = (pd.DataFrame(df_tmp, columns=['date','smooth_us10yBE'])
                        .set_index('date').resample('M').last())
# Plot OOS Breakeven Inflation
plot_factor(global_inflation.reset_index(), 'smooth_us10yBE', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
# Build ranges
global_inflation['z'] = (
    (global_inflation['smooth_us10yBE'] - global_inflation['smooth_us10yBE'].ewm(halflife=12,min_periods=12).mean()) /
    global_inflation['smooth_us10yBE'].ewm(halflife=12,min_periods=12).std()
    ) * (12**0.5)
# Build trend
global_inflation['trend'] = 'Neutral'
idx = global_inflation['z'] > 1
global_inflation.loc[idx, 'trend'] = 'Uptrend'
idx = global_inflation['z'] < -1
global_inflation.loc[idx, 'trend'] = 'Downtrend'
# Build signal
global_inflation['global_inflation_signal'] = global_inflation['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
global_inflation['global_inflation_signal'] = global_inflation['global_inflation_signal'].astype(int)
# Combine with returns
global_inflation_factor = pd.concat([global_inflation, monthly_data], axis=1).sort_index().dropna()
global_inflation_factor['next_monthly_total_return'] = global_inflation_factor['monthly_total_return'].shift(-1)
global_inflation_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=global_inflation_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((global_inflation_factor['global_inflation_signal'] * global_inflation_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### US Dollar Cycle

In [ ]:
dollar_variables = ['dollar_index_dxy']
ylabel = 'Dollar Factor'
nickname = 'dollar_cycle'
dollar_cycle = econ_data2.query("indicator in @dollar_variables").pivot_table('value','date','indicator').sort_index()
dollar_cycle = dollar_cycle.query('date >= @date_start and date <= @date_end').sort_index().copy()
# Smooth data with HP filter (OOS)
df_tmp = []
idx = dollar_cycle.resample('M').last().index
for i, dt in enumerate(idx[12:]):
    trend = (hpfilter(dollar_cycle[['dollar_index_dxy']]
                      .query("date <= @dt"), lamb=129_600)[1].iloc[-1]
            )
    df_tmp.append((dt, trend))
dollar_cycle = (pd.DataFrame(df_tmp, columns=['date','dollar_index_dxy'])
                .set_index('date').resample('M').last())
# Plot OOS Breakeven Inflation
plot_factor(dollar_cycle.reset_index(), 'dollar_index_dxy', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
# Build ranges
dollar_cycle['z'] = (
    (dollar_cycle['dollar_index_dxy'] - dollar_cycle['dollar_index_dxy'].ewm(halflife=12,min_periods=12).mean()) /
    dollar_cycle['dollar_index_dxy'].ewm(halflife=12,min_periods=12).std()
    ) * (12**0.5)
# Build trend
dollar_cycle['trend'] = 'Neutral'
idx = dollar_cycle['z'] > 1
dollar_cycle.loc[idx, 'trend'] = 'Uptrend'
idx = dollar_cycle['z'] < -1
dollar_cycle.loc[idx, 'trend'] = 'Downtrend'
# Build signal
dollar_cycle['dollar_cycle_signal'] = dollar_cycle['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
dollar_cycle['dollar_cycle_signal'] = dollar_cycle['dollar_cycle_signal'].astype(int)
# Combine with returns
dollar_cycle_factor = pd.concat([dollar_cycle, monthly_data], axis=1).sort_index().dropna()
dollar_cycle_factor['next_monthly_total_return'] = dollar_cycle_factor['monthly_total_return'].shift(-1)
dollar_cycle_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=dollar_cycle_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((dollar_cycle_factor['dollar_cycle_signal'] * dollar_cycle_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Global Financial Risk

In [ ]:
global_fin_risk_variables = ['ofr_fsi']
ylabel = 'Global Financial Risk Factor'
nickname = 'global_fin_risk'

global_fin_risk = econ_data2.query("indicator in @global_fin_risk_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
global_fin_risk = global_fin_risk.query('date >= @date_start and date <= @date_end').copy()
global_fin_risk.tail(5)

In [ ]:
# Build ranges
global_fin_risk['z'] = (
    (global_fin_risk['ofr_fsi'] - global_fin_risk['ofr_fsi'].ewm(halflife=36,min_periods=12).mean()) /
    global_fin_risk['ofr_fsi'].ewm(halflife=36,min_periods=12).std()
    ) * (12**0.5)
# Build trend
global_fin_risk['trend'] = 'Neutral'
idx = global_fin_risk['z'] > 1
global_fin_risk.loc[idx, 'trend'] = 'Uptrend'
idx = global_fin_risk['z'] < -1
global_fin_risk.loc[idx, 'trend'] = 'Downtrend'
# Build signal
global_fin_risk['fin_risk_signal'] = global_fin_risk['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
global_fin_risk['fin_risk_signal'] = global_fin_risk['fin_risk_signal'].astype(int)
# Combine with returns
global_fin_risk_factor = pd.concat([global_fin_risk, monthly_data], axis=1).sort_index().dropna()
global_fin_risk_factor['next_monthly_total_return'] = global_fin_risk_factor['monthly_total_return'].shift(-1)
global_fin_risk_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=global_fin_risk_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((global_fin_risk_factor['fin_risk_signal'] * global_fin_risk_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Autocorrelation

In [ ]:
monthly_data['monthly_total_return']

In [ ]:
ylabel = 'Auto Correlation Factor'
ar_factor = walk_forward_ar_backtest(monthly_data['monthly_total_return'].iloc[1:], n_ar=12, initial_window=36).to_frame('ar_factor')
# Plot OOS Factor
plot_factor(ar_factor.apply(np.sign).reset_index(), 'ar_factor', f"{ylabel} Time Series", 'date', ylabel)
ar_factor

In [ ]:
ar_factor['ar_signal'] = ar_factor['ar_factor'].apply(np.sign)
ar_factor['trend'] = ar_factor['ar_signal'].map({-1:'Downtrend', 0:'Neutral', 1:'Uptrend'})
# Combine with returns
ar_factor = pd.concat([ar_factor, monthly_data], axis=1).sort_index().dropna()
ar_factor['next_monthly_total_return'] = ar_factor['monthly_total_return'].shift(-1)
ar_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=ar_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((ar_factor['ar_signal'] * ar_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

### Backtest

In [ ]:
df_signal = pd.concat([
                    dom_econ_factor['dom_econ_signal'], 
                    dom_curncy_factor['dom_curncy_signal'], 
                    dom_credit_factor['dom_credit_signal'], 
                    dom_inflation_factor['dom_inflation_signal'],
                    global_econ_factor['global_econ_signal'],
                    global_curncy_factor['global_curncy_signal'],
                    global_inflation['global_inflation_signal'],
                    dollar_cycle_factor['dollar_cycle_signal'],
                    global_fin_risk_factor['fin_risk_signal'],
                    ar_factor['ar_signal']
                    ], axis=1)

df_weight = {
            'dom_econ_signal': -1,
            'dom_curncy_signal': 1,
            'dom_credit_signal': 1,
            'dom_inflation_signal': 1,
            'global_econ_signal': -1,
            'global_curncy_signal': 1,
            'global_inflation_signal': -1,
            'dollar_cycle_signal': 1,
            'fin_risk_signal': 1,
            'ar_signal': 1,
            }


df_signal = df_signal.loc[df_signal.count(axis=1)>df_signal.shape[1]//2].copy()
df_pos = (df_signal * pd.Series(df_weight)).mean(axis=1) / pd.Series(df_weight).abs().sum()
#df_strat = df_pos.apply(np.sign).shift(1) * monthly_data['monthly_total_return']
df_strat = df_pos.shift(1) * monthly_data['monthly_total_return']
print(f"{ir(df_strat, fnorm=12, precision=2).to_frame('Sharpe Ratio')}")
_ = df_strat.add(1).cumprod().plot()

In [ ]:
df_signal.to_csv('all_signal.csv')

#### Clustermap

In [ ]:
sns.clustermap(df_signal.corr(), cmap='viridis', standard_scale=1, figsize=(6,6))
plt.show()

In [ ]:
df_pos = pd.DataFrame(df_pos)

In [ ]:
df_pos['up'] = (df_signal * pd.Series(df_weight)).mean(axis=1) 
df_pos['down'] =  pd.Series(df_weight).abs().sum()

In [ ]:
df_pos.to_csv('df_pos.csv')

In [ ]:
df_pos